# 03 — Download Face models (SCRFD + ArcFace, CPU)

Uses **InsightFace** pack **`buffalo_s`** (smaller, CPU-friendly):

| Component | Role |
|-----------|------|
| SCRFD | Face detection (near + far better than Haar) |
| ArcFace | Face embedding for recognition |
| landmarks / gender age | bundled (optional) |

Files land under `models/face/models/buffalo_s/` (InsightFace layout).


In [1]:
from pathlib import Path

def find_project_root(start: Path | None = None) -> Path:
    start = (start or Path.cwd()).resolve()
    for candidate in (start, *start.parents):
        if (candidate / "main.py").exists() and (candidate / "src").exists():
            return candidate
    raise FileNotFoundError("Could not find the project root from the current notebook location.")

ROOT = find_project_root()
FACE_ROOT = ROOT / "models" / "face"
FACE_ROOT.mkdir(parents=True, exist_ok=True)
print("InsightFace root:", FACE_ROOT)


InsightFace root: /Users/macbookpro/Desktop/person-face-events/models/face


In [ ]:
# %pip install insightface onnxruntime opencv-python numpy tqdm


In [2]:
from insightface.app import FaceAnalysis
from pathlib import Path

FACE_ROOT = str(ROOT / "models" / "face")
PACK = "buffalo_l"   # use buffalo_l only if CPU is strong and you need max accuracy

print("Downloading / preparing", PACK, "into", FACE_ROOT)
print("(First run may take several minutes...)")

app = FaceAnalysis(
    name=PACK,
    root=FACE_ROOT,
    providers=["CPUExecutionProvider"],
)
# ctx_id=-1 => CPU
app.prepare(ctx_id=-1, det_size=(640, 640))
print("✓ Face models ready")

# List downloaded files
pack_dir = Path(FACE_ROOT) / "models" / PACK
if pack_dir.exists():
    for p in sorted(pack_dir.rglob("*")):
        if p.is_file():
            print(f"  {p.relative_to(FACE_ROOT)}  ({p.stat().st_size/1e6:.1f} MB)")
else:
    print("Pack dir not found yet — check InsightFace cache under models/face")


KeyboardInterrupt: 

In [ ]:
# Smoke test: detect + embed on a blank image (should return 0 faces, but API works)
import numpy as np
import cv2
from insightface.app import FaceAnalysis
from pathlib import Path

FACE_ROOT = str(ROOT / "models" / "face")
app = FaceAnalysis(name="buffalo_s", root=FACE_ROOT, providers=["CPUExecutionProvider"])
app.prepare(ctx_id=-1, det_size=(640, 640))

img = np.zeros((480, 640, 3), dtype=np.uint8)
# draw a simple skin-colored oval so detector might still return 0 — just test call
faces = app.get(img)
print("Faces on blank frame:", len(faces))
print("CPU provider OK")

# If you have a selfie path, test it:
# img = cv2.imread("/path/to/face.jpg")
# faces = app.get(img)
# print(len(faces), faces[0].normed_embedding.shape if faces else None)


## Config

In `config/settings.yaml`:

```yaml
models:
  face_root: "models/face"
  face_pack: "buffalo_s"
  face_match_threshold: 0.42
  min_face_px: 40
```

Enroll people:

```bash
# data/faces_gallery/Ahmed/*.jpg
python scripts/enroll_face.py
```
